# Harmony Regression Test Suite

This test suite ensures some Harmony Service was deployed successfully and the expected endpoints are returning successfully.

In [ ]:
harmony_host_url = 'https://harmony.uat.earthdata.nasa.gov'

## Prerequisites

The dependencies for this notebook are listed in the [environment.yaml](./environment.yaml). To test or install locally, create the papermill environment used in the automated regression testing suite:

`conda env create -f ./environment.yaml && conda activate papermill-harmony`

A `.netrc` file must also be located in the `test` directory of this repository.

## Ensure endpoints are returning expected values.

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Set up retrying so we don't fail for a transient error.
retry_strategy = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[500, 502, 503, 504],
)

adapter = HTTPAdapter(max_retries=retry_strategy)
session = requests.Session()
session.mount('https://', adapter)
session.mount('http://', adapter)

harmony_endpoints = [
    '',
    'cloud-access',
    'docs',
    'docs/api',
    'docs/edr-api',
    'health',
    'jobs',
    'readiness',
    'service-image-tag',
    'versions',
    'workflow-ui',
]

for endpoint in harmony_endpoints:
    try:
        response = session.get(f'{harmony_host_url}/{endpoint}')
        response.raise_for_status()
        assert response.status_code == 200
        print(f'Endpoint /{endpoint} alive.')
    except Exception:
        raise Exception(
            f'TEST FAILED: Bad response from endpoint {harmony_host_url}/{endpoint}'
        )

# /capabilities requires a collection so a 400 error is expected.
response = session.get(f'{harmony_host_url}/capabilities')
assert response.status_code == 400
print('Endpoint /capabilities alive.')


print('All expected endpoints responding.')